# Import Data and Cleaning

In [1]:
# Import libraries
import numpy as np
import pandas as pd
import time

from sklearn import datasets, metrics, svm, tree
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV, Ridge, RidgeCV, Lasso, LassoCV
from sklearn.metrics import confusion_matrix, roc_curve, auc, roc_auc_score, accuracy_score, classification_report, recall_score, precision_score, f1_score
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, ComplementNB, CategoricalNB
from sklearn.preprocessing import StandardScaler, MinMaxScaler, FunctionTransformer
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier

import lightgbm as lgb

In [ ]:
# # Import libraries
# import numpy as np
# import pandas as pd
# import time

# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import confusion_matrix, roc_auc_score, classification_report, recall_score, precision_score, f1_score, accuracy_score
# from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
# from sklearn.preprocessing import FunctionTransformer, StandardScaler
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.pipeline import Pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.utils.class_weight import compute_sample_weight

# import lightgbm as lgb

In [2]:
df = pd.read_csv("data/online_shopping_cleaned.csv")
df

,CustomerType,SpecialDayProximity,ExitRate,PageValue,TrafficSource,GeographicRegion,BounceRate,ProductPageTime,PurchaseCompleted
0,Returning_Visitor,0.0,0.200000,0.000000,1.0,1,0.200000,0.000000,0
1,Returning_Visitor,0.0,0.100000,0.000000,2.0,1,0.000000,64.000000,0
2,Returning_Visitor,NaN,0.200000,0.000000,3.0,9,0.200000,0.000000,0
3,Returning_Visitor,0.0,0.140000,0.000000,4.0,2,0.050000,2.666667,0
4,Returning_Visitor,0.0,NaN,NaN,4.0,1,0.020000,627.500000,0
...,...,...,...,...,...,...,...,...,...
12325,Returning_Visitor,0.0,0.029031,12.241717,NaN,1,0.007143,1783.791667,0
12326,Returning_Visitor,0.0,0.021333,NaN,8.0,1,0.000000,465.750000,0
12327,Returning_Visitor,0.0,0.086667,0.000000,13.0,1,0.083333,184.250000,0
12328,Returning_Visitor,0.0,0.021053,0.000000,11.0,3,0.000000,346.000000,0


In [3]:
df.drop_duplicates(inplace=True)
df

,CustomerType,SpecialDayProximity,ExitRate,PageValue,TrafficSource,GeographicRegion,BounceRate,ProductPageTime,PurchaseCompleted
0,Returning_Visitor,0.0,0.200000,0.000000,1.0,1,0.200000,0.000000,0
1,Returning_Visitor,0.0,0.100000,0.000000,2.0,1,0.000000,64.000000,0
2,Returning_Visitor,NaN,0.200000,0.000000,3.0,9,0.200000,0.000000,0
3,Returning_Visitor,0.0,0.140000,0.000000,4.0,2,0.050000,2.666667,0
4,Returning_Visitor,0.0,NaN,NaN,4.0,1,0.020000,627.500000,0
...,...,...,...,...,...,...,...,...,...
12325,Returning_Visitor,0.0,0.029031,12.241717,NaN,1,0.007143,1783.791667,0
12326,Returning_Visitor,0.0,0.021333,NaN,8.0,1,0.000000,465.750000,0
12327,Returning_Visitor,0.0,0.086667,0.000000,13.0,1,0.083333,184.250000,0
12328,Returning_Visitor,0.0,0.021053,0.000000,11.0,3,0.000000,346.000000,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
Index: 11897 entries, 0 to 12329
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CustomerType         11897 non-null  str    
 1   SpecialDayProximity  11287 non-null  float64
 2   ExitRate             11290 non-null  float64
 3   PageValue            11291 non-null  float64
 4   TrafficSource        11301 non-null  float64
 5   GeographicRegion     11897 non-null  int64  
 6   BounceRate           11897 non-null  float64
 7   ProductPageTime      11297 non-null  float64
 8   PurchaseCompleted    11897 non-null  int64  
dtypes: float64(6), int64(2), str(1)
memory usage: 929.5 KB


In [5]:
df.describe(include="all")

,CustomerType,SpecialDayProximity,ExitRate,PageValue,TrafficSource,GeographicRegion,BounceRate,ProductPageTime,PurchaseCompleted
count,11897,11287.000000,11290.000000,11291.000000,11301.000000,11897.000000,11897.000000,11297.000000,11897.000000
unique,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,Returning_Visitor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,9612,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,0.062426,0.037316,6.127517,4.068313,3.158191,0.015950,1242.965989,0.160377
std,NaN,0.199977,0.038846,18.993455,4.000567,2.401039,0.035825,1950.087213,0.366970
min,NaN,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000
25%,NaN,0.000000,0.013810,0.000000,2.000000,1.000000,0.000000,217.761111,0.000000
50%,NaN,0.000000,0.025000,0.000000,2.000000,3.000000,0.002469,646.791667,0.000000
75%,NaN,0.000000,0.044444,0.000000,4.000000,4.000000,0.014815,1522.816667,0.000000


In [6]:
df.dropna(inplace=True)
df

,CustomerType,SpecialDayProximity,ExitRate,PageValue,TrafficSource,GeographicRegion,BounceRate,ProductPageTime,PurchaseCompleted
0,Returning_Visitor,0.0,0.200000,0.0,1.0,1,0.200000,0.000000,0
1,Returning_Visitor,0.0,0.100000,0.0,2.0,1,0.000000,64.000000,0
3,Returning_Visitor,0.0,0.140000,0.0,4.0,2,0.050000,2.666667,0
5,Returning_Visitor,0.0,0.024561,0.0,3.0,1,0.015789,154.216667,0
6,Returning_Visitor,0.4,0.200000,0.0,3.0,3,0.200000,0.000000,0
...,...,...,...,...,...,...,...,...,...
12323,Returning_Visitor,0.0,0.013953,0.0,10.0,1,0.000000,1157.976190,0
12324,Returning_Visitor,0.0,0.037647,0.0,1.0,1,0.000000,503.000000,0
12327,Returning_Visitor,0.0,0.086667,0.0,13.0,1,0.083333,184.250000,0
12328,Returning_Visitor,0.0,0.021053,0.0,11.0,3,0.000000,346.000000,0


In [7]:
df.info()

<class 'pandas.DataFrame'>
Index: 9177 entries, 0 to 12329
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CustomerType         9177 non-null   str    
 1   SpecialDayProximity  9177 non-null   float64
 2   ExitRate             9177 non-null   float64
 3   PageValue            9177 non-null   float64
 4   TrafficSource        9177 non-null   float64
 5   GeographicRegion     9177 non-null   int64  
 6   BounceRate           9177 non-null   float64
 7   ProductPageTime      9177 non-null   float64
 8   PurchaseCompleted    9177 non-null   int64  
dtypes: float64(6), int64(2), str(1)
memory usage: 717.0 KB


In [8]:
df.describe(include="all")

,CustomerType,SpecialDayProximity,ExitRate,PageValue,TrafficSource,GeographicRegion,BounceRate,ProductPageTime,PurchaseCompleted
count,9177,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000
unique,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,Returning_Visitor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,7401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,0.062243,0.036462,6.220592,4.050561,3.152991,0.014793,1244.123949,0.162144
std,NaN,0.200013,0.037447,19.291962,3.979877,2.398216,0.033065,1969.727163,0.368603
min,NaN,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000
25%,NaN,0.000000,0.013441,0.000000,2.000000,1.000000,0.000000,221.780000,0.000000
50%,NaN,0.000000,0.024815,0.000000,2.000000,3.000000,0.002174,654.066667,0.000000
75%,NaN,0.000000,0.044351,0.000000,4.000000,4.000000,0.014286,1527.583333,0.000000


# Feature Engineering

In [9]:
# New column: has_page_value
df["has_page_value"] = df["PageValue"] > 0
df["has_page_value"] = df["has_page_value"].astype(int)
df["has_page_value"].value_counts()

has_page_value
0    7050
1    2127
Name: count, dtype: int64

In [10]:
# Log transform ProductPageTime and PageValue
log_transformer = FunctionTransformer(np.log1p, validate=True)
df["PageValue_log"] = log_transformer.transform(df[["PageValue"]].values)
df["ProductPageTime_log"] = log_transformer.transform(df[["ProductPageTime"]].values)
df.describe(include='all')

,CustomerType,SpecialDayProximity,ExitRate,PageValue,TrafficSource,GeographicRegion,BounceRate,ProductPageTime,PurchaseCompleted,has_page_value,PageValue_log,ProductPageTime_log
count,9177,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000,9177.000000
unique,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,Returning_Visitor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,7401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,0.062243,0.036462,6.220592,4.050561,3.152991,0.014793,1244.123949,0.162144,0.231775,0.654797,6.208250
std,NaN,0.200013,0.037447,19.291962,3.979877,2.398216,0.033065,1969.727163,0.368603,0.421989,1.291921,1.685346
min,NaN,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,NaN,0.000000,0.013441,0.000000,2.000000,1.000000,0.000000,221.780000,0.000000,0.000000,0.000000,5.406185
50%,NaN,0.000000,0.024815,0.000000,2.000000,3.000000,0.002174,654.066667,0.000000,0.000000,0.000000,6.484737
75%,NaN,0.000000,0.044351,0.000000,4.000000,4.000000,0.014286,1527.583333,0.000000,0.000000,0.000000,7.332097


# One-hot encode

In [196]:
# For linear models to avoid collinearity
df_drop_first = pd.get_dummies(df, columns=["CustomerType", "TrafficSource", "GeographicRegion"], drop_first=True, dtype='int')
df_drop_first

,SpecialDayProximity,ExitRate,PageValue,BounceRate,ProductPageTime,PurchaseCompleted,has_page_value,PageValue_log,ProductPageTime_log,CustomerType_Other,...,TrafficSource_19.0,TrafficSource_20.0,GeographicRegion_2,GeographicRegion_3,GeographicRegion_4,GeographicRegion_5,GeographicRegion_6,GeographicRegion_7,GeographicRegion_8,GeographicRegion_9
0,0.0,0.200000,0.0,0.200000,0.000000,0,0,0.0,0.000000,0,...,0,0,0,0,0,0,0,0,0,0
1,0.0,0.100000,0.0,0.000000,64.000000,0,0,0.0,4.174387,0,...,0,0,0,0,0,0,0,0,0,0
3,0.0,0.140000,0.0,0.050000,2.666667,0,0,0.0,1.299283,0,...,0,0,1,0,0,0,0,0,0,0
5,0.0,0.024561,0.0,0.015789,154.216667,0,0,0.0,5.044822,0,...,0,0,0,0,0,0,0,0,0,0
6,0.4,0.200000,0.0,0.200000,0.000000,0,0,0.0,0.000000,0,...,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12323,0.0,0.013953,0.0,0.000000,1157.976190,0,0,0.0,7.055292,0,...,0,0,0,0,0,0,0,0,0,0
12324,0.0,0.037647,0.0,0.000000,503.000000,0,0,0.0,6.222576,0,...,0,0,0,0,0,0,0,0,0,0
12327,0.0,0.086667,0.0,0.083333,184.250000,0,0,0.0,5.221706,0,...,0,0,0,0,0,0,0,0,0,0
12328,0.0,0.021053,0.0,0.000000,346.000000,0,0,0.0,5.849325,0,...,0,0,0,1,0,0,0,0,0,0


# Set Predictors and Target

In [197]:
X = df_drop_first.drop(columns='PurchaseCompleted')
# X = df.drop(columns='PurchaseCompleted')  # For lightgbm (categorical) - no one-hot encoding 
X

,SpecialDayProximity,ExitRate,PageValue,BounceRate,ProductPageTime,has_page_value,PageValue_log,ProductPageTime_log,CustomerType_Other,CustomerType_Returning_Visitor,...,TrafficSource_19.0,TrafficSource_20.0,GeographicRegion_2,GeographicRegion_3,GeographicRegion_4,GeographicRegion_5,GeographicRegion_6,GeographicRegion_7,GeographicRegion_8,GeographicRegion_9
0,0.0,0.200000,0.0,0.200000,0.000000,0,0.0,0.000000,0,1,...,0,0,0,0,0,0,0,0,0,0
1,0.0,0.100000,0.0,0.000000,64.000000,0,0.0,4.174387,0,1,...,0,0,0,0,0,0,0,0,0,0
3,0.0,0.140000,0.0,0.050000,2.666667,0,0.0,1.299283,0,1,...,0,0,1,0,0,0,0,0,0,0
5,0.0,0.024561,0.0,0.015789,154.216667,0,0.0,5.044822,0,1,...,0,0,0,0,0,0,0,0,0,0
6,0.4,0.200000,0.0,0.200000,0.000000,0,0.0,0.000000,0,1,...,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12323,0.0,0.013953,0.0,0.000000,1157.976190,0,0.0,7.055292,0,1,...,0,0,0,0,0,0,0,0,0,0
12324,0.0,0.037647,0.0,0.000000,503.000000,0,0.0,6.222576,0,1,...,0,0,0,0,0,0,0,0,0,0
12327,0.0,0.086667,0.0,0.083333,184.250000,0,0.0,5.221706,0,1,...,0,0,0,0,0,0,0,0,0,0
12328,0.0,0.021053,0.0,0.000000,346.000000,0,0.0,5.849325,0,1,...,0,0,0,1,0,0,0,0,0,0


In [198]:
y = df.PurchaseCompleted
y

0        0
1        0
3        0
5        0
6        0
        ..
12323    0
12324    0
12327    0
12328    0
12329    0
Name: PurchaseCompleted, Length: 9177, dtype: int64

# Split Data

**Use stratify to keep proportions of y the same.**

In [199]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 1, stratify = y)
print(X_train.shape, "\n", X_train[:5], "\n")
print(X_test.shape, "\n", X_test[:5], "\n")
print(y_train.shape, "\n", y_train[:5], "\n")
print(y_test.shape, "\n", y_test[:5])

(7341, 38) 
       SpecialDayProximity  ExitRate  PageValue  BounceRate  ProductPageTime  \
5429                  0.0  0.116000   0.000000    0.080000       334.000000   
6015                  0.0  0.009375   0.000000    0.004167      3025.333333   
9563                  0.0  0.019022  10.519018    0.002020      2434.144872   
5241                  0.6  0.028571   0.000000    0.000000       486.000000   
4205                  0.0  0.058333   0.000000    0.011111       110.000000   

      has_page_value  PageValue_log  ProductPageTime_log  CustomerType_Other  \
5429               0       0.000000             5.814131                   0   
6015               0       0.000000             8.015107                   0   
9563               1       2.443999             7.797762                   0   
5241               0       0.000000             6.188264                   0   
4205               0       0.000000             4.709530                   0   

      CustomerType_Returning_Vi

In [200]:
y_train.value_counts(normalize=True)

PurchaseCompleted
0    0.837897
1    0.162103
Name: proportion, dtype: float64

In [201]:
y_test.value_counts(normalize=True)

PurchaseCompleted
0    0.837691
1    0.162309
Name: proportion, dtype: float64

# Normalise Data

In [166]:
# Apply StandardScaler to continuous features: ExitRate, PageValue, BounceRate, ProductPageTime (leave SpecialDayProximity as is - it's already bounded between 0 and 1)

# scale_features = ['ExitRate', 'PageValue', 'BounceRate', 'ProductPageTime']

# With feature engineered features
scale_features = ['ExitRate', 'PageValue', 'BounceRate', 'ProductPageTime', 'PageValue_log', 'ProductPageTime_log']

passthrough_features = X.drop(columns=scale_features).columns.tolist()

scaler = ColumnTransformer(
    transformers=[
        ('scale', StandardScaler(), scale_features)
    ],
    remainder='passthrough'
)

In [167]:
# fit transform X_train
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=scale_features+passthrough_features)

binary_features = passthrough_features.copy()
binary_features.remove('SpecialDayProximity')  # Continuous feature

# Convert binary features back to int type
for col in binary_features:
    X_train_scaled[col] = X_train_scaled[col].astype('int')

X_train_scaled

,ExitRate,PageValue,BounceRate,ProductPageTime,PageValue_log,ProductPageTime_log,SpecialDayProximity,has_page_value,CustomerType_Other,CustomerType_Returning_Visitor,...,TrafficSource_19.0,TrafficSource_20.0,GeographicRegion_2,GeographicRegion_3,GeographicRegion_4,GeographicRegion_5,GeographicRegion_6,GeographicRegion_7,GeographicRegion_8,GeographicRegion_9
0,-0.632774,-0.313690,-0.276196,-0.298494,-0.492037,0.166850,0.0,0,0,1,...,0,0,0,1,0,0,0,0,0,0
1,-0.783959,-0.313690,-0.449549,-0.275177,-0.492037,0.205481,0.0,0,0,1,...,0,0,0,0,0,0,1,0,0,0
2,-0.376715,-0.313690,-0.449549,-0.396778,-0.492037,-0.029647,0.0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,3.135770,-0.313690,2.514797,-0.466675,-0.492037,-0.221599,0.0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,0.130109,-0.313690,-0.023637,0.225238,-0.492037,0.718678,0.8,0,0,1,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9471,-0.798002,0.478033,-0.449549,0.035279,1.688338,0.570137,0.0,1,0,1,...,0,0,0,0,0,0,0,0,1,0
9472,-0.859726,2.881459,-0.449549,0.378805,2.754792,0.816972,0.0,1,0,1,...,0,0,0,1,0,0,0,0,0,0
9473,2.433273,-0.313690,2.020739,-0.583482,-0.492037,-0.806644,0.0,0,0,1,...,0,0,1,0,0,0,0,0,0,0
9474,-0.118676,-0.313690,-0.296221,0.430859,-0.492037,0.847005,0.0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [168]:
# Only transform X_test
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=scale_features+passthrough_features)

for col in binary_features:
    X_test_scaled[col] = X_test_scaled[col].astype('int')

X_test_scaled

,ExitRate,PageValue,BounceRate,ProductPageTime,PageValue_log,ProductPageTime_log,SpecialDayProximity,has_page_value,CustomerType_Other,CustomerType_Returning_Visitor,...,TrafficSource_19.0,TrafficSource_20.0,GeographicRegion_2,GeographicRegion_3,GeographicRegion_4,GeographicRegion_5,GeographicRegion_6,GeographicRegion_7,GeographicRegion_8,GeographicRegion_9
0,-0.487781,-0.313690,-0.449549,-0.493827,-0.492037,-0.316114,0.0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
1,-0.608214,-0.313690,-0.449549,0.198236,-0.492037,0.699636,0.0,0,0,1,...,0,0,0,0,0,1,0,0,0,0
2,-0.651521,0.122671,-0.364854,-0.167113,1.256458,0.358238,0.0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,-0.877626,-0.313690,-0.449549,-0.453986,-0.492037,-0.181981,0.4,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,0.845019,-0.313690,-0.449549,-0.642491,-0.492037,-1.818428,0.0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2364,-0.584409,-0.313690,-0.251926,0.018457,-0.492037,0.555065,0.0,0,0,1,...,0,0,0,1,0,0,0,0,0,0
2365,-0.682148,0.703709,-0.449549,-0.311357,1.875752,0.144447,0.0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
2366,-0.291872,-0.313690,-0.083581,0.457569,-0.492037,0.861862,0.6,0,0,1,...,0,0,0,1,0,0,0,0,0,0
2367,-0.513205,-0.313690,-0.378970,0.836630,-0.492037,1.041150,0.0,0,0,1,...,0,0,0,1,0,0,0,0,0,0


# Modelling

In [169]:
# helper function to evaluate model performance
def evaluate(model):
    y_pred_class = model.predict(X_test_scaled)
    y_pred_prob = model.predict_proba(X_test_scaled)
    accuracy = model.score(X_test_scaled, y_test)
    roc_auc = roc_auc_score(y_test, y_pred_prob[:,1])
    f1 = f1_score(y_test, y_pred_class)
    
    print("Train Accuracy: ", model.score(X_train_scaled, y_train))
    print("Test Accuracy: ", accuracy)
    print("confusion_matrix:\n", confusion_matrix(y_test, y_pred_class))
    print("roc_auc_score: ", roc_auc)
    print("classification_report:\n", classification_report(y_test, y_pred_class))
    
    return accuracy, recall_score(y_test, y_pred_class), precision_score(y_test, y_pred_class), roc_auc, f1

In [156]:
# Non-scaled
def evaluate_(model):
    y_pred_class = model.predict(X_test)
    y_pred_prob = model.predict_proba(X_test)
    accuracy = model.score(X_test, y_test)
    roc_auc = roc_auc_score(y_test, y_pred_prob[:,1])
    f1 = f1_score(y_test, y_pred_class)
    
    print("Train Accuracy: ", model.score(X_train, y_train))
    print("Test Accuracy: ", accuracy)
    print("confusion_matrix:\n", confusion_matrix(y_test, y_pred_class))
    print("roc_auc_score: ", roc_auc)
    print("classification_report:\n", classification_report(y_test, y_pred_class))
    
    return accuracy, recall_score(y_test, y_pred_class), precision_score(y_test, y_pred_class), roc_auc, f1

In [170]:
results_df = pd.DataFrame(columns = ['accuracy', 'recall', 'precision', 'f1_score', 'roc_auc', 'time_taken (s)'])

# helper function to save results
def save_results(model_name, accuracy, recall, precision, f1, roc_auc, time_taken):
    results_df.loc[model_name] = {'accuracy': accuracy, 'recall': recall, 'precision': precision, 'f1_score': f1, 'roc_auc': roc_auc, 'time_taken (s)': time_taken}

## Logistic Regression

In [171]:
start_time = time.perf_counter()
lr = LogisticRegression(max_iter=1000, random_state=1).fit(X_train_scaled, y_train)
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(lr)
save_results('Logistic Regression', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.8862389193752638
Test Accuracy:  0.8877163360067539
confusion_matrix:
 [[1886  101]
 [ 165  217]]
roc_auc_score:  0.8828352880108136
classification_report:
               precision    recall  f1-score   support

           0       0.92      0.95      0.93      1987
           1       0.68      0.57      0.62       382

    accuracy                           0.89      2369
   macro avg       0.80      0.76      0.78      2369
weighted avg       0.88      0.89      0.88      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.887716,0.568063,0.68239,0.62,0.882835,0.118498


In [172]:
start_time = time.perf_counter()
lr = LogisticRegression(max_iter=1000, random_state=1, class_weight='balanced').fit(X_train_scaled, y_train)  # class_weight='balanced'
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(lr)
save_results('Logistic Regression (balanced)', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.8653440270156184
Test Accuracy:  0.8704094554664416
confusion_matrix:
 [[1770  217]
 [  90  292]]
roc_auc_score:  0.8840354977510889
classification_report:
               precision    recall  f1-score   support

           0       0.95      0.89      0.92      1987
           1       0.57      0.76      0.66       382

    accuracy                           0.87      2369
   macro avg       0.76      0.83      0.79      2369
weighted avg       0.89      0.87      0.88      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.887716,0.568063,0.682390,0.620000,0.882835,0.118498
Logistic Regression (balanced),0.870409,0.764398,0.573674,0.655443,0.884035,0.157189


## SVM

In [173]:
start_time = time.perf_counter()
svm_model = svm.SVC(probability=True, random_state=1).fit(X_train_scaled, y_train)  
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(svm_model)
save_results('SVM', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df


Train Accuracy:  0.8968974250738708
Test Accuracy:  0.8961587167581258
confusion_matrix:
 [[1911   76]
 [ 170  212]]
roc_auc_score:  0.8526008057610067
classification_report:
               precision    recall  f1-score   support

           0       0.92      0.96      0.94      1987
           1       0.74      0.55      0.63       382

    accuracy                           0.90      2369
   macro avg       0.83      0.76      0.79      2369
weighted avg       0.89      0.90      0.89      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.887716,0.568063,0.682390,0.620000,0.882835,0.118498
Logistic Regression (balanced),0.870409,0.764398,0.573674,0.655443,0.884035,0.157189
SVM,0.896159,0.554974,0.736111,0.632836,0.852601,32.829640


In [174]:
start_time = time.perf_counter()
svm_model = svm.SVC(probability=True, random_state=1, class_weight='balanced').fit(X_train_scaled, y_train)  # class_weight='balanced'  
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(svm_model)
save_results('SVM (balanced)', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.8626002532714225
Test Accuracy:  0.8623891937526382
confusion_matrix:
 [[1749  238]
 [  88  294]]
roc_auc_score:  0.8720452575246959
classification_report:
               precision    recall  f1-score   support

           0       0.95      0.88      0.91      1987
           1       0.55      0.77      0.64       382

    accuracy                           0.86      2369
   macro avg       0.75      0.82      0.78      2369
weighted avg       0.89      0.86      0.87      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.887716,0.568063,0.682390,0.620000,0.882835,0.118498
Logistic Regression (balanced),0.870409,0.764398,0.573674,0.655443,0.884035,0.157189
SVM,0.896159,0.554974,0.736111,0.632836,0.852601,32.829640
SVM (balanced),0.862389,0.769634,0.552632,0.643326,0.872045,46.697763


## Naive Bayes

In [175]:
start_time = time.perf_counter()
gaussian_nb = GaussianNB().fit(X_train_scaled, y_train)
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(gaussian_nb)
save_results('Gaussian NB', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.2514774166314901
Test Accuracy:  0.24947235120303926
confusion_matrix:
 [[ 212 1775]
 [   3  379]]
roc_auc_score:  0.8522740746791316
classification_report:
               precision    recall  f1-score   support

           0       0.99      0.11      0.19      1987
           1       0.18      0.99      0.30       382

    accuracy                           0.25      2369
   macro avg       0.58      0.55      0.25      2369
weighted avg       0.86      0.25      0.21      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.887716,0.568063,0.682390,0.620000,0.882835,0.118498
Logistic Regression (balanced),0.870409,0.764398,0.573674,0.655443,0.884035,0.157189
SVM,0.896159,0.554974,0.736111,0.632836,0.852601,32.829640
SVM (balanced),0.862389,0.769634,0.552632,0.643326,0.872045,46.697763
Gaussian NB,0.249472,0.992147,0.175952,0.298896,0.852274,0.024033


In [176]:
start_time = time.perf_counter()
bernoulli_nb = BernoulliNB().fit(X_train_scaled, y_train)
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(bernoulli_nb)
save_results('Bernoulli NB', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.8708315745040102
Test Accuracy:  0.8746306458421275
confusion_matrix:
 [[1788  199]
 [  98  284]]
roc_auc_score:  0.8770483272159086
classification_report:
               precision    recall  f1-score   support

           0       0.95      0.90      0.92      1987
           1       0.59      0.74      0.66       382

    accuracy                           0.87      2369
   macro avg       0.77      0.82      0.79      2369
weighted avg       0.89      0.87      0.88      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.887716,0.568063,0.682390,0.620000,0.882835,0.118498
Logistic Regression (balanced),0.870409,0.764398,0.573674,0.655443,0.884035,0.157189
SVM,0.896159,0.554974,0.736111,0.632836,0.852601,32.829640
SVM (balanced),0.862389,0.769634,0.552632,0.643326,0.872045,46.697763
Gaussian NB,0.249472,0.992147,0.175952,0.298896,0.852274,0.024033
Bernoulli NB,0.874631,0.743455,0.587992,0.656647,0.877048,0.025867


## Bagging Models

In [177]:
start_time = time.perf_counter()
rf = RandomForestClassifier(random_state=1).fit(X_train_scaled, y_train)
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(rf)
save_results('Random Forest', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.9995778809624314
Test Accuracy:  0.8906711692697341
confusion_matrix:
 [[1895   92]
 [ 167  215]]
roc_auc_score:  0.8638084723477473
classification_report:
               precision    recall  f1-score   support

           0       0.92      0.95      0.94      1987
           1       0.70      0.56      0.62       382

    accuracy                           0.89      2369
   macro avg       0.81      0.76      0.78      2369
weighted avg       0.88      0.89      0.89      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.887716,0.568063,0.682390,0.620000,0.882835,0.118498
Logistic Regression (balanced),0.870409,0.764398,0.573674,0.655443,0.884035,0.157189
SVM,0.896159,0.554974,0.736111,0.632836,0.852601,32.829640
SVM (balanced),0.862389,0.769634,0.552632,0.643326,0.872045,46.697763
Gaussian NB,0.249472,0.992147,0.175952,0.298896,0.852274,0.024033
Bernoulli NB,0.874631,0.743455,0.587992,0.656647,0.877048,0.025867
Random Forest,0.890671,0.562827,0.700326,0.624093,0.863808,2.667846


In [178]:
start_time = time.perf_counter()
rf = RandomForestClassifier(random_state=1, class_weight='balanced').fit(X_train_scaled, y_train)  # class_weight='balanced'
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(rf)
save_results('Random Forest (balanced)', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.9995778809624314
Test Accuracy:  0.8889826931194597
confusion_matrix:
 [[1896   91]
 [ 172  210]]
roc_auc_score:  0.8635680351604804
classification_report:
               precision    recall  f1-score   support

           0       0.92      0.95      0.94      1987
           1       0.70      0.55      0.61       382

    accuracy                           0.89      2369
   macro avg       0.81      0.75      0.78      2369
weighted avg       0.88      0.89      0.88      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.887716,0.568063,0.682390,0.620000,0.882835,0.118498
Logistic Regression (balanced),0.870409,0.764398,0.573674,0.655443,0.884035,0.157189
SVM,0.896159,0.554974,0.736111,0.632836,0.852601,32.829640
SVM (balanced),0.862389,0.769634,0.552632,0.643326,0.872045,46.697763
Gaussian NB,0.249472,0.992147,0.175952,0.298896,0.852274,0.024033
Bernoulli NB,0.874631,0.743455,0.587992,0.656647,0.877048,0.025867
Random Forest,0.890671,0.562827,0.700326,0.624093,0.863808,2.667846
Random Forest (balanced),0.888983,0.549738,0.697674,0.614934,0.863568,2.835958


## Boosting Models

### Gradient Boosting

In [179]:
start_time = time.perf_counter()
gb = GradientBoostingClassifier(random_state=1).fit(X_train_scaled, y_train)
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(gb)
save_results('Gradient Boosting', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.9057619248628113
Test Accuracy:  0.897002954833263
confusion_matrix:
 [[1908   79]
 [ 165  217]]
roc_auc_score:  0.8907730088507234
classification_report:
               precision    recall  f1-score   support

           0       0.92      0.96      0.94      1987
           1       0.73      0.57      0.64       382

    accuracy                           0.90      2369
   macro avg       0.83      0.76      0.79      2369
weighted avg       0.89      0.90      0.89      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.887716,0.568063,0.682390,0.620000,0.882835,0.118498
Logistic Regression (balanced),0.870409,0.764398,0.573674,0.655443,0.884035,0.157189
SVM,0.896159,0.554974,0.736111,0.632836,0.852601,32.829640
SVM (balanced),0.862389,0.769634,0.552632,0.643326,0.872045,46.697763
Gaussian NB,0.249472,0.992147,0.175952,0.298896,0.852274,0.024033
Bernoulli NB,0.874631,0.743455,0.587992,0.656647,0.877048,0.025867
Random Forest,0.890671,0.562827,0.700326,0.624093,0.863808,2.667846
Random Forest (balanced),0.888983,0.549738,0.697674,0.614934,0.863568,2.835958
Gradient Boosting,0.897003,0.568063,0.733108,0.640118,0.890773,3.949655


In [180]:
weights = compute_sample_weight(class_weight='balanced', y=y_train)

start_time = time.perf_counter()
gb = GradientBoostingClassifier(random_state=1).fit(X_train_scaled, y_train, sample_weight=weights)  # Introduce class weights as GradientBoostingClassifier does not have a built-in class_weight parameter
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(gb)
save_results('Gradient Boosting (balanced)', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.8670325031658928
Test Accuracy:  0.8611228366399325
confusion_matrix:
 [[1742  245]
 [  84  298]]
roc_auc_score:  0.8918605754155939
classification_report:
               precision    recall  f1-score   support

           0       0.95      0.88      0.91      1987
           1       0.55      0.78      0.64       382

    accuracy                           0.86      2369
   macro avg       0.75      0.83      0.78      2369
weighted avg       0.89      0.86      0.87      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.887716,0.568063,0.682390,0.620000,0.882835,0.118498
Logistic Regression (balanced),0.870409,0.764398,0.573674,0.655443,0.884035,0.157189
SVM,0.896159,0.554974,0.736111,0.632836,0.852601,32.829640
SVM (balanced),0.862389,0.769634,0.552632,0.643326,0.872045,46.697763
Gaussian NB,0.249472,0.992147,0.175952,0.298896,0.852274,0.024033
Bernoulli NB,0.874631,0.743455,0.587992,0.656647,0.877048,0.025867
Random Forest,0.890671,0.562827,0.700326,0.624093,0.863808,2.667846
Random Forest (balanced),0.888983,0.549738,0.697674,0.614934,0.863568,2.835958
Gradient Boosting,0.897003,0.568063,0.733108,0.640118,0.890773,3.949655
Gradient Boosting (balanced),0.861123,0.780105,0.548803,0.644324,0.891861,3.800717


### XGBoost

In [181]:
start_time = time.perf_counter()
xgb_model = XGBClassifier(objective='binary:logistic', eval_metric='aucpr', random_state=1).fit(X_train_scaled, y_train)
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(xgb_model)
save_results('XGBoost', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.9585268045588856
Test Accuracy:  0.8915154073448712
confusion_matrix:
 [[1904   83]
 [ 174  208]]
roc_auc_score:  0.8867633070455342
classification_report:
               precision    recall  f1-score   support

           0       0.92      0.96      0.94      1987
           1       0.71      0.54      0.62       382

    accuracy                           0.89      2369
   macro avg       0.82      0.75      0.78      2369
weighted avg       0.88      0.89      0.89      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.887716,0.568063,0.682390,0.620000,0.882835,0.118498
Logistic Regression (balanced),0.870409,0.764398,0.573674,0.655443,0.884035,0.157189
SVM,0.896159,0.554974,0.736111,0.632836,0.852601,32.829640
SVM (balanced),0.862389,0.769634,0.552632,0.643326,0.872045,46.697763
Gaussian NB,0.249472,0.992147,0.175952,0.298896,0.852274,0.024033
Bernoulli NB,0.874631,0.743455,0.587992,0.656647,0.877048,0.025867
Random Forest,0.890671,0.562827,0.700326,0.624093,0.863808,2.667846
Random Forest (balanced),0.888983,0.549738,0.697674,0.614934,0.863568,2.835958
Gradient Boosting,0.897003,0.568063,0.733108,0.640118,0.890773,3.949655
Gradient Boosting (balanced),0.861123,0.780105,0.548803,0.644324,0.891861,3.800717


In [182]:
scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1])

start_time = time.perf_counter()
xgb_model = XGBClassifier(objective='binary:logistic', eval_metric='aucpr', random_state=1, scale_pos_weight=scale_pos_weight).fit(X_train_scaled, y_train)  # "balanced" approach for XGBoost
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(xgb_model)
save_results('XGBoost (balanced)', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.943858168003377
Test Accuracy:  0.8632334318277755
confusion_matrix:
 [[1775  212]
 [ 112  270]]
roc_auc_score:  0.8753514335326217
classification_report:
               precision    recall  f1-score   support

           0       0.94      0.89      0.92      1987
           1       0.56      0.71      0.62       382

    accuracy                           0.86      2369
   macro avg       0.75      0.80      0.77      2369
weighted avg       0.88      0.86      0.87      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.887716,0.568063,0.682390,0.620000,0.882835,0.118498
Logistic Regression (balanced),0.870409,0.764398,0.573674,0.655443,0.884035,0.157189
SVM,0.896159,0.554974,0.736111,0.632836,0.852601,32.829640
SVM (balanced),0.862389,0.769634,0.552632,0.643326,0.872045,46.697763
Gaussian NB,0.249472,0.992147,0.175952,0.298896,0.852274,0.024033
Bernoulli NB,0.874631,0.743455,0.587992,0.656647,0.877048,0.025867
Random Forest,0.890671,0.562827,0.700326,0.624093,0.863808,2.667846
Random Forest (balanced),0.888983,0.549738,0.697674,0.614934,0.863568,2.835958
Gradient Boosting,0.897003,0.568063,0.733108,0.640118,0.890773,3.949655
Gradient Boosting (balanced),0.861123,0.780105,0.548803,0.644324,0.891861,3.800717


### LightGBM

In [183]:
start_time = time.perf_counter()
lgbm = lgb.LGBMClassifier(objective='binary', random_state=1, verbose=0).fit(X_train_scaled, y_train)
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(lgbm)
save_results('LightGBM', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.9374208526804559
Test Accuracy:  0.8944702406078514
confusion_matrix:
 [[1909   78]
 [ 172  210]]
roc_auc_score:  0.8897480218277442
classification_report:
               precision    recall  f1-score   support

           0       0.92      0.96      0.94      1987
           1       0.73      0.55      0.63       382

    accuracy                           0.89      2369
   macro avg       0.82      0.76      0.78      2369
weighted avg       0.89      0.89      0.89      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.887716,0.568063,0.682390,0.620000,0.882835,0.118498
Logistic Regression (balanced),0.870409,0.764398,0.573674,0.655443,0.884035,0.157189
SVM,0.896159,0.554974,0.736111,0.632836,0.852601,32.829640
SVM (balanced),0.862389,0.769634,0.552632,0.643326,0.872045,46.697763
Gaussian NB,0.249472,0.992147,0.175952,0.298896,0.852274,0.024033
Bernoulli NB,0.874631,0.743455,0.587992,0.656647,0.877048,0.025867
Random Forest,0.890671,0.562827,0.700326,0.624093,0.863808,2.667846
Random Forest (balanced),0.888983,0.549738,0.697674,0.614934,0.863568,2.835958
Gradient Boosting,0.897003,0.568063,0.733108,0.640118,0.890773,3.949655
Gradient Boosting (balanced),0.861123,0.780105,0.548803,0.644324,0.891861,3.800717


In [184]:
start_time = time.perf_counter()
lgbm = lgb.LGBMClassifier(objective='binary', random_state=1, verbose=0, class_weight='balanced').fit(X_train_scaled, y_train)  # class_weight='balanced'
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate(lgbm)
save_results('LightGBM (balanced)', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.8972140143520473
Test Accuracy:  0.8623891937526382
confusion_matrix:
 [[1751  236]
 [  90  292]]
roc_auc_score:  0.8861783530118547
classification_report:
               precision    recall  f1-score   support

           0       0.95      0.88      0.91      1987
           1       0.55      0.76      0.64       382

    accuracy                           0.86      2369
   macro avg       0.75      0.82      0.78      2369
weighted avg       0.89      0.86      0.87      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression,0.887716,0.568063,0.682390,0.620000,0.882835,0.118498
Logistic Regression (balanced),0.870409,0.764398,0.573674,0.655443,0.884035,0.157189
SVM,0.896159,0.554974,0.736111,0.632836,0.852601,32.829640
SVM (balanced),0.862389,0.769634,0.552632,0.643326,0.872045,46.697763
Gaussian NB,0.249472,0.992147,0.175952,0.298896,0.852274,0.024033
Bernoulli NB,0.874631,0.743455,0.587992,0.656647,0.877048,0.025867
Random Forest,0.890671,0.562827,0.700326,0.624093,0.863808,2.667846
Random Forest (balanced),0.888983,0.549738,0.697674,0.614934,0.863568,2.835958
Gradient Boosting,0.897003,0.568063,0.733108,0.640118,0.890773,3.949655
Gradient Boosting (balanced),0.861123,0.780105,0.548803,0.644324,0.891861,3.800717


### LightGBM (Categorical)

In [153]:
for col in ['CustomerType', 'TrafficSource', 'GeographicRegion', 'SpecialDayProximity']:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

X_train.info()

<class 'pandas.DataFrame'>
Index: 9476 entries, 10685 to 9603
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   CustomerType         9476 non-null   category
 1   SpecialDayProximity  9476 non-null   category
 2   ExitRate             9476 non-null   float64 
 3   PageValue            9476 non-null   float64 
 4   TrafficSource        9476 non-null   category
 5   GeographicRegion     9476 non-null   category
 6   BounceRate           9476 non-null   float64 
 7   ProductPageTime      9476 non-null   float64 
dtypes: category(4), float64(4)
memory usage: 407.5 KB


In [157]:
start_time = time.perf_counter()
lgbm = lgb.LGBMClassifier(objective='binary', random_state=1, verbose=0).fit(X_train, y_train)
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc, f1 = evaluate_(lgbm)
save_results('LightGBM', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.9373153229210638
Test Accuracy:  0.8927817644575771
confusion_matrix:
 [[1906   81]
 [ 173  209]]
roc_auc_score:  0.8837983542239215
classification_report:
               precision    recall  f1-score   support

           0       0.92      0.96      0.94      1987
           1       0.72      0.55      0.62       382

    accuracy                           0.89      2369
   macro avg       0.82      0.75      0.78      2369
weighted avg       0.89      0.89      0.89      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
LightGBM,0.892782,0.54712,0.72069,0.622024,0.883798,0.295459


In [158]:
start_time = time.perf_counter()
lgbm = lgb.LGBMClassifier(objective='binary', random_state=1, verbose=0, class_weight='balanced').fit(X_train, y_train)  # class_weight='balanced'
end_time = time.perf_counter()
execution_time = end_time - start_time
accuracy, recall, precision, roc_auc,f1 = evaluate_(lgbm)
save_results('LightGBM (balanced)', accuracy, recall, precision, f1, roc_auc, execution_time)
results_df

Train Accuracy:  0.8965808357956944
Test Accuracy:  0.8636555508653441
confusion_matrix:
 [[1752  235]
 [  88  294]]
roc_auc_score:  0.8888804717575234
classification_report:
               precision    recall  f1-score   support

           0       0.95      0.88      0.92      1987
           1       0.56      0.77      0.65       382

    accuracy                           0.86      2369
   macro avg       0.75      0.83      0.78      2369
weighted avg       0.89      0.86      0.87      2369



,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
LightGBM,0.892782,0.547120,0.720690,0.622024,0.883798,0.295459
LightGBM (balanced),0.863656,0.769634,0.555766,0.645445,0.888880,0.250852


# Prelimiary Results

## Cleaned (dropna)

In [86]:
# df_cleaned (dropna) - 8 main features, scaled
results_df_one_hot_encoded = results_df.copy()
results_df_one_hot_encoded.sort_values(by='f1_score', ascending=False)

,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Gradient Boosting (balanced),0.862200,0.828859,0.550111,0.661312,0.907713,2.542727
SVM (balanced),0.868192,0.755034,0.571066,0.650289,0.880495,26.330326
Logistic Regression (balanced),0.873094,0.721477,0.589041,0.648567,0.892247,0.091949
LightGBM (balanced),0.858932,0.795302,0.544828,0.646658,0.903219,0.270334
XGBoost (balanced),0.866558,0.708054,0.571816,0.632684,0.888514,0.394017
Gradient Boosting,0.892702,0.560403,0.716738,0.629002,0.911509,2.439249
LightGBM,0.890523,0.570470,0.699588,0.628466,0.903041,0.267017
XGBoost,0.885076,0.550336,0.680498,0.608534,0.894195,0.596488
Bernoulli NB,0.877996,0.570470,0.639098,0.602837,0.859996,0.022613
Random Forest,0.882898,0.523490,0.681223,0.592030,0.887765,1.616100


In [98]:
# LightGBM (Categorical)
results_df_lightgbm = results_df.copy()
results_df_lightgbm

,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
LightGBM,0.889978,0.563758,0.700000,0.624535,0.905848,0.471160
LightGBM (balanced),0.865468,0.802013,0.559719,0.659310,0.901763,0.751011


In [124]:
# df_cleaned (dropna) - 8 main features, scaled, Feature engineered
results_df_one_hot_encoded_feature_eng = results_df.copy()
results_df_one_hot_encoded_feature_eng.sort_values(by='f1_score', ascending=False)

,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Logistic Regression (balanced),0.873638,0.822148,0.577830,0.678670,0.899093,0.119151
Bernoulli NB,0.873094,0.805369,0.578313,0.673212,0.893661,0.035806
SVM (balanced),0.863834,0.825503,0.554054,0.663073,0.892491,25.316269
Gradient Boosting (balanced),0.862200,0.828859,0.550111,0.661312,0.908038,3.090376
LightGBM (balanced),0.864924,0.808725,0.557870,0.660274,0.903936,0.306869
Gradient Boosting,0.893246,0.563758,0.717949,0.631579,0.911570,3.219405
LightGBM,0.892157,0.563758,0.711864,0.629213,0.902747,0.238243
XGBoost (balanced),0.863290,0.697987,0.563686,0.623688,0.890512,0.324080
Random Forest,0.886710,0.570470,0.680000,0.620438,0.890999,2.797184
Random Forest (balanced),0.889434,0.546980,0.705628,0.616257,0.890329,2.252027


## Fillna

In [150]:
# Fillna - 8 main features, scaled
results_df_fillna_one_hot_encoded = results_df.copy()
results_df_fillna_one_hot_encoded.sort_values(by='f1_score', ascending=False)

,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
LightGBM,0.898269,0.578534,0.734219,0.647145,0.890902,0.215508
Gradient Boosting (balanced),0.861123,0.780105,0.548803,0.644324,0.891846,3.030748
SVM (balanced),0.867033,0.743455,0.566866,0.643262,0.860145,46.452437
Gradient Boosting,0.897003,0.568063,0.733108,0.640118,0.890753,3.098834
LightGBM (balanced),0.859856,0.761780,0.546992,0.636761,0.887408,0.221694
XGBoost (balanced),0.866188,0.704188,0.568710,0.629240,0.874019,0.309654
Logistic Regression (balanced),0.863656,0.698953,0.562105,0.623104,0.874365,0.118996
XGBoost,0.890671,0.544503,0.709898,0.616296,0.885089,0.401235
Random Forest (balanced),0.890249,0.536649,0.711806,0.611940,0.856557,2.468983
Bernoulli NB,0.879696,0.583770,0.638968,0.610123,0.856035,0.021059


In [159]:
# Fillna - LightGBM (Categorical)
results_df_lightgbm = results_df.copy()
results_df_lightgbm

,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
LightGBM,0.892782,0.547120,0.720690,0.622024,0.883798,0.295459
LightGBM (balanced),0.863656,0.769634,0.555766,0.645445,0.888880,0.250852


In [185]:
# Fillna - Scaled, Feature engineered
results_df_fillna_one_hot_encoded_feature_eng = results_df.copy()
results_df_fillna_one_hot_encoded_feature_eng.sort_values(by='f1_score', ascending=False)

,accuracy,recall,precision,f1_score,roc_auc,time_taken (s)
Bernoulli NB,0.874631,0.743455,0.587992,0.656647,0.877048,0.025867
Logistic Regression (balanced),0.870409,0.764398,0.573674,0.655443,0.884035,0.157189
Gradient Boosting (balanced),0.861123,0.780105,0.548803,0.644324,0.891861,3.800717
SVM (balanced),0.862389,0.769634,0.552632,0.643326,0.872045,46.697763
LightGBM (balanced),0.862389,0.764398,0.553030,0.641758,0.886178,0.346028
Gradient Boosting,0.897003,0.568063,0.733108,0.640118,0.890773,3.949655
SVM,0.896159,0.554974,0.736111,0.632836,0.852601,32.829640
LightGBM,0.894470,0.549738,0.729167,0.626866,0.889748,0.240577
XGBoost (balanced),0.863233,0.706806,0.560166,0.625000,0.875351,0.390198
Random Forest,0.890671,0.562827,0.700326,0.624093,0.863808,2.667846


**Observations**
- Dropna consistently outperforms fillna. Possibly due to:
    - Fillna's median/mode imputations create artificial noise - introduced bias, distort correlations, and artificially boost confidence.
    - Dropna having less artificial noise, ensures models train on real data, which preserves original data variance and feature relationships.

**Follow-up**
- Proceed to finetune using dropna dataset, using the following 3 models:

| Model | Strategy | Rationale |
|-------|----------|-----------|
| **Logistic Regression** | Linear, regularized (ElasticNet) | Interpretable baseline. ElasticNet penalty with tunable L1 ratio provides built-in feature selection. Fast to train, sets a performance floor. |
| **LightGBM** | Gradient boosting (sequential ensemble) | Sequentially corrects errors from previous trees. Typically achieves the best predictive performance on tabular data. Handles feature interactions natively. |
| **Random Forest** | Bagging (parallel ensemble) | Aggregates many independent decision trees via bagging. Robust to overfitting, handles non-linear relationships. Provides a contrast to the boosting approach of LightGBM. |

This selection covers **linear vs. bagging vs. boosting** — three distinct paradigms — allowing meaningful comparison. All models are configured with `class_weight='balanced'` to handle the ~16.2% minority class (after cleaning).

# Finetuning

Using cleaned / dropna dataset.

In [204]:
# Continue right after train-test split, without scaling - as scaling is handled within pipelines and cross-validation to avoid data leakage

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

scale_features = ['ExitRate', 'PageValue', 'BounceRate', 'ProductPageTime', 'PageValue_log', 'ProductPageTime_log']

preprocessor = ColumnTransformer(
    transformers=[('scale', StandardScaler(), scale_features)],
    remainder='passthrough'
).set_output(transform="pandas")

# =====================
# Logistic Regression
# =====================
lr_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(class_weight='balanced', max_iter=5000, random_state=1))
])

lr_params = {
    'model__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'model__l1_ratio': [0, 0.25, 0.5, 0.75, 1],  # 0 = L2, 1 = L1
    'model__solver': ['saga'],
}

lr_search = GridSearchCV(lr_pipe, lr_params, cv=cv, scoring='f1', n_jobs=-1)
lr_search.fit(X_train, y_train)

# =====================
# LightGBM
# =====================
lgbm_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', lgb.LGBMClassifier(objective='binary', class_weight='balanced', random_state=1, verbose=-1))
])

lgbm_params = {
    'model__n_estimators': [100, 200, 500],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__max_depth': [3, 5, 7, -1],
    'model__num_leaves': [15, 31, 63],
    'model__min_child_samples': [5, 10, 20],
    'model__subsample': [0.7, 0.8, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 1.0],
}

lgbm_search = RandomizedSearchCV(lgbm_pipe, lgbm_params, n_iter=50, cv=cv, scoring='f1', n_jobs=-1, random_state=1)
lgbm_search.fit(X_train, y_train)

# =====================
# Random Forest
# =====================
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(class_weight='balanced', random_state=1))
])

rf_params = {
    'model__n_estimators': [100, 200, 500],
    'model__max_depth': [5, 10, 15, None],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2'],
}

rf_search = RandomizedSearchCV(rf_pipe, rf_params, n_iter=50, cv=cv, scoring='f1', n_jobs=-1, random_state=1)
rf_search.fit(X_train, y_train)

# =====================
# Compare results (df_cleaned, dropna)
# =====================
for name, search in [('LR', lr_search), ('LightGBM', lgbm_search), ('RF', rf_search)]:
    y_pred = search.best_estimator_.predict(X_test)
    y_prob = search.best_estimator_.predict_proba(X_test)[:, 1]
    print(f"\n{name}:")
    print(f"  Best CV F1:  {search.best_score_:.4f}")
    print(f"  Accuracy:    {accuracy_score(y_test, y_pred):.4f}")
    print(f"  Precision:   {precision_score(y_test, y_pred):.4f}")
    print(f"  Recall:      {recall_score(y_test, y_pred):.4f}")
    print(f"  F1:          {f1_score(y_test, y_pred):.4f}")
    print(f"  AUC-ROC:     {roc_auc_score(y_test, y_prob):.4f}")
    print(f"  Best params: {search.best_params_}")


LR:
  Best CV F1:  0.6714
  Accuracy:    0.8742
  Precision:   0.5857
  Recall:      0.7685
  F1:          0.6647
  AUC-ROC:     0.8706
  Best params: {'model__C': 0.001, 'model__l1_ratio': 1, 'model__solver': 'saga'}

LightGBM:
  Best CV F1:  0.6677
  Accuracy:    0.8687
  Precision:   0.5652
  Recall:      0.8289
  F1:          0.6721
  AUC-ROC:     0.9130
  Best params: {'model__subsample': 1.0, 'model__num_leaves': 31, 'model__n_estimators': 100, 'model__min_child_samples': 20, 'model__max_depth': 3, 'model__learning_rate': 0.05, 'model__colsample_bytree': 1.0}

RF:
  Best CV F1:  0.6740
  Accuracy:    0.8818
  Precision:   0.6092
  Recall:      0.7584
  F1:          0.6756
  AUC-ROC:     0.9048
  Best params: {'model__n_estimators': 200, 'model__min_samples_split': 2, 'model__min_samples_leaf': 2, 'model__max_features': 'log2', 'model__max_depth': None}


In [205]:
  # =====================
  # Feature Importance
  # =====================
results = {'LR': lr_search, 'LightGBM': lgbm_search, 'RF': rf_search}

for name, search in results.items():
    best = search.best_estimator_
    model = best.named_steps['model']
    feature_names = best.named_steps['preprocessor'].get_feature_names_out()

    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    elif hasattr(model, 'coef_'):
        importances = np.abs(model.coef_[0])

    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False).reset_index(drop=True)

    print(f"\n{name} — Top 10 Features:")
    for i, row in importance_df.head(10).iterrows():
        print(f"  {i+1:>2}. {row['feature']:<45} {row['importance']:.4f}")


LR — Top 10 Features:
   1. scale__PageValue_log                          0.8250
   2. scale__ExitRate                               0.0000
   3. scale__PageValue                              0.0000
   4. scale__BounceRate                             0.0000
   5. scale__ProductPageTime                        0.0000
   6. scale__ProductPageTime_log                    0.0000
   7. remainder__SpecialDayProximity                0.0000
   8. remainder__has_page_value                     0.0000
   9. remainder__CustomerType_Other                 0.0000
  10. remainder__CustomerType_Returning_Visitor     0.0000

LightGBM — Top 10 Features:
   1. scale__ExitRate                               112.0000
   2. scale__PageValue                              104.0000
   3. scale__ProductPageTime                        78.0000
   4. scale__BounceRate                             75.0000
   5. scale__ProductPageTime_log                    72.0000
   6. remainder__SpecialDayProximity                53.0

**Note**

RandomForestClassifier gives a different set of results when running multiple times. This is likely n_jobs=-1 floating-point non-determinism: parallel CV fold evaluation can accumulate floating-point values in slightly different orders, tipping close hyperparameter combinations differently. LR (GridSearch, small space) and LightGBM are less sensitive to this.

Example:
```
Model                     Accuracy Precision   Recall       F1  AUC-ROC    CV F1
--------------------------------------------------------------------------------
Random Forest               0.8834    0.6207   0.7248   0.6687   0.9003   0.6730

RF — Top 10 Features:
     1. scale__PageValue_log                          0.2076
     2. scale__PageValue                              0.1947
     3. remainder__has_page_value                     0.1507
     4. scale__ExitRate                               0.0921
     5. scale__ProductPageTime_log                    0.0907
     6. scale__ProductPageTime                        0.0891
     7. scale__BounceRate                             0.0567
     8. remainder__TrafficSource_2.0                  0.0136
     9. remainder__SpecialDayProximity                0.0102
    10. remainder__CustomerType_Returning_Visitor     0.0099
```

# End of Notebook